In [39]:
import pandas as pd
import numpy as np

df=pd.read_csv('workout_fitness_tracker_data.csv')
df.head()

,User ID,Age,Gender,Height (cm),Weight (kg),Workout Type,Workout Duration (mins),Calories Burned,Heart Rate (bpm),Steps Taken,Distance (km),Workout Intensity,Sleep Hours,Water Intake (liters),Daily Calories Intake,Resting Heart Rate (bpm),VO2 Max,Body Fat (%),Mood Before Workout,Mood After Workout
0,1,39,Male,175,99,Cycling,79,384,112,8850,14.44,High,8.2,1.9,3195,61,38.4,28.5,Tired,Fatigued
1,2,36,Other,157,112,Cardio,73,612,168,2821,1.10,High,8.6,1.9,2541,73,38.4,28.5,Happy,Energized
2,3,25,Female,180,66,HIIT,27,540,133,18898,7.28,High,9.8,1.9,3362,80,38.4,28.5,Happy,Fatigued
3,4,56,Male,154,89,Cycling,39,672,118,14102,6.55,Medium,5.8,1.9,2071,65,38.4,28.5,Neutral,Neutral
4,5,53,Other,194,59,Strength,56,410,170,16518,3.17,Medium,7.3,1.9,3298,59,38.4,28.5,Stressed,Energized


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   User ID                   10000 non-null  int64  
 1   Age                       10000 non-null  int64  
 2   Gender                    10000 non-null  object 
 3   Height (cm)               10000 non-null  int64  
 4   Weight (kg)               10000 non-null  int64  
 5   Workout Type              10000 non-null  object 
 6   Workout Duration (mins)   10000 non-null  int64  
 7   Calories Burned           10000 non-null  int64  
 8   Heart Rate (bpm)          10000 non-null  int64  
 9   Steps Taken               10000 non-null  int64  
 10  Distance (km)             10000 non-null  float64
 11  Workout Intensity         10000 non-null  object 
 12  Sleep Hours               10000 non-null  float64
 13  Water Intake (liters)     10000 non-null  float64
 14  Daily C

In [41]:
import pandas as pd
import numpy as np

train_df=pd.read_csv('train.txt',sep=';',header=None,names=['text','label'])
test_df=pd.read_csv('test.txt',sep=';',header=None,names=['text','label'])
val_df=pd.read_csv('val.txt',sep=';',header=None,names=['text','label'])
train_df.head()
test_df.head()
val_df.head()

,text,label
0,im feeling quite sad and sorry for myself but ...,sadness
1,i feel like i am still looking at a blank canv...,sadness
2,i feel like a faithful servant,love
3,i am just feeling cranky and blue,anger
4,i can have for a treat or if i am feeling festive,joy


In [42]:
import re
def clean(text):
    text=text.lower()
    text=text.strip()
    text=re.sub(r'#','',text)
    text=re.sub(r'@\w+','',text)
    text=re.sub(r'https?://\S+','',text)
    text=re.sub(r'[^\w\s]','',text)
    text=re.sub(r'\s+',' ',text)
    return text

train_df['text']=train_df['text'].apply(clean)
test_df['text']=test_df['text'].apply(clean)
val_df['text']=val_df['text'].apply(clean)

In [43]:
from collections import Counter

words=[word for text in train_df['text'] for word in text.split()]
counts=Counter(words)
vocab={"<PAD>": 0,"<UNK>":1}
vocab.update({w:i+2 for i,(w, _) in enumerate(counts.items())})
maxlength=50
def encodeandpad(text):
    ids=[vocab.get(w, 1) for w in text.split()]
    return (ids+[0]*maxlength)[:maxlength]
for df in (train_df,val_df,test_df):
    df['padded']=df['text'].apply(encodeandpad)

In [44]:
train_df.head()

,text,label,padded
0,i didnt feel humiliated,sadness,"[2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,i can go from feeling so hopeless to so damned...,sadness,"[2, 6, 7, 8, 9, 10, 11, 12, 10, 13, 14, 15, 8,..."
2,im grabbing a minute to post i feel greedy wrong,anger,"[24, 25, 26, 27, 12, 28, 2, 4, 29, 30, 0, 0, 0..."
3,i am ever feeling nostalgic about the fireplac...,love,"[2, 31, 32, 9, 33, 34, 35, 36, 2, 37, 38, 39, ..."
4,i am feeling grouchy,anger,"[2, 31, 9, 44, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."


In [45]:
import torch
from torch.utils.data import Dataset,DataLoader

labels=sorted(train_df['label'].unique())
labelmap={label:idx for idx,label in enumerate(labels)}
reverseforlabels={v:k for k,v in labelmap.items()}
class dataset(Dataset):
    def __init__(self,df):
        self.X=torch.tensor(df['padded'].tolist(),dtype=torch.long)
        self.y=torch.tensor(df['label'].map(labelmap).tolist(),dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

train_loader=DataLoader(dataset(train_df),batch_size=32, shuffle=True)
val_loader=DataLoader(dataset(val_df),batch_size=32)
test_loader=DataLoader(dataset(test_df),batch_size=32)

In [46]:
class RNN(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(RNN,self).__init__()
        self.embedding=torch.nn.Embedding(vocab_size,embedding_dim)
        self.lstm=torch.nn.LSTM(embedding_dim, hidden_dim,batch_first=True)
        self.fc=torch.nn.Linear(hidden_dim,output_dim)
        self.softmax=torch.nn.LogSoftmax(dim=1)

    def forward(self,x):
        x=self.embedding(x)
        x,_=self.lstm(x)
        out=self.fc(x[:,-1])
        return self.softmax(out)
    
vocabsize=len(vocab)
embed=100
hidden=128
outputdim=len(labelmap)
model=RNN(vocabsize, embed, hidden, outputdim)




In [47]:
from sklearn.metrics import accuracy_score, classification_report

criterion=torch.nn.NLLLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
epochs=5
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        X,y=batch
        optimizer.zero_grad()
        output=model(X)
        loss=criterion(output,y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

Epoch 1/5, Loss: 1.6248081922531128
Epoch 2/5, Loss: 1.5610564947128296
Epoch 3/5, Loss: 1.6457982063293457
Epoch 4/5, Loss: 1.2740696668624878
Epoch 5/5, Loss: 1.1548373699188232


In [48]:
model.eval()
valloss=0
for epoch in range(epochs):
    with torch.no_grad():
        for batch in val_loader:
            X,y=batch
            output=model(X)
            loss=criterion(output,y)
            valloss+=loss.item()
    print(f"Validation Loss: {valloss/len(val_loader)}")

Validation Loss: 1.2760578270942446
Validation Loss: 2.5521156541884893
Validation Loss: 3.8281734812827337
Validation Loss: 5.104231308376979
Validation Loss: 6.380289135471223


In [49]:
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")
model.eval()
ytrue,ypred=[],[]
with torch.no_grad():
    for batch in test_loader:
        X,y=batch
        output=model(X)
        ytrue.extend(y.tolist())
        ypred.extend(output.argmax(dim=1).tolist())
print(classification_report(ytrue,ypred))
print("test data acc",accuracy_score(ytrue, ypred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       275
           1       0.00      0.00      0.00       224
           2       0.59      0.87      0.70       695
           3       0.00      0.00      0.00       159
           4       0.56      0.94      0.70       581
           5       0.00      0.00      0.00        66

    accuracy                           0.57      2000
   macro avg       0.19      0.30      0.23      2000
weighted avg       0.37      0.57      0.45      2000

test data acc 0.574


In [52]:

def predicttext(text):
    cleaned=clean(text)
    encoded=encodeandpad(cleaned)
    tensor=torch.tensor([encoded],dtype=torch.long)
    with torch.no_grad():
        output=model(tensor)
        pred = torch.argmax(output, dim=1).item()
    return reverseforlabels[pred]
print("sample pred")
for text,truelabel in zip(test_df['text'][:5],test_df['label'][:5]):
    predicted = predicttext(text)
    print(f"Text: {text}\nActual: {truelabel} Predicted: {predicted}\n")

sample pred
Text: im feeling rather rotten so im not very ambitious right now
Actual: sadness Predicted: sadness

Text: im updating my blog because i feel shitty
Actual: sadness Predicted: sadness

Text: i never make her separate from me because i don t ever want her to feel like i m ashamed with her
Actual: sadness Predicted: sadness

Text: i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived
Actual: joy Predicted: joy

Text: i was feeling a little vain when i did this one
Actual: sadness Predicted: sadness

